# Stronger nature-inspired re-run - ALL models (Phase 3 NOT required)

Runs the **stronger** Phase-4 nature-inspired search (PSO/GWO/SA with a bigger budget and higher proxy fidelity) followed by a full Phase-5 retrain, for every model in the `MODELS` list. It loads each model's committed Phase-3 baseline from `results/<model>_report.json` for the before/after comparison, so **the baseline is never retrained**. New weights only replace the deployed checkpoint if they beat the baseline.

### How to run
1. *Add Input* -> the prepped dataset published by `00_prepare_dataset` (set `PREPPED_INPUT` if the slug differs).
2. *Settings -> Accelerator -> GPU*.
3. Run the setup cell, then the re-run cell. Edit `MODELS` to run a subset per session if you hit the time limit.

Outputs: `artifacts/<model>_report_rerun.json` and `artifacts/checkpoints/<model>_rerun.pt`. Note: `ssdlite` has no committed Phase-3 baseline yet, so its before/after shows `None` until you run the full `06_ssdlite.ipynb` once (it still trains and saves a re-run checkpoint here).

In [ ]:
import os, sys, shutil, glob

# ===== EDIT IF NEEDED =====
REPO_URL = "https://github.com/202201638/Graduation_project_Fully_team_2026.git"
# Folder you attached from the 00_prepare_dataset output (the prepped yolo_dataset).
PREPPED_INPUT = "/kaggle/input/rsna-prepped-yolo-dataset"
# ==========================

WORK = "/kaggle/working"
os.environ["YOLO_DATASET_DIR"] = f"{WORK}/yolo_dataset"
os.environ["ARTIFACT_DIR"]     = f"{WORK}/artifacts"
os.environ["RUNS_DIR"]         = f"{WORK}/runs"
os.environ["PNG_DIR"]          = f"{WORK}/png_images"

REPO_DIR = f"{WORK}/repo"
if not os.path.exists(REPO_DIR):
    os.system(f"GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {REPO_URL} {REPO_DIR}")
AI_DIR = f"{REPO_DIR}/ai"
sys.path.insert(0, AI_DIR)
os.chdir(AI_DIR)

# deps that may be missing / outdated on Kaggle (torch/torchvision already present)
os.system("pip install -q ultralytics torchmetrics pydicom")

# Locate the prepped dataset regardless of how Kaggle nested it, then copy it into the
# writable working dir (Ultralytics writes .cache files next to labels, which fails on
# the read-only /kaggle/input).
print("Attached inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")

def _find_yolo_root(base):
    cands = [base, os.path.join(base, "yolo_dataset")]
    cands += glob.glob(os.path.join(base, "*", "yolo_dataset"))
    cands += glob.glob(os.path.join(base, "*"))
    for c in cands:
        if os.path.isdir(os.path.join(c, "train", "images")):
            return c
    return None

SRC = _find_yolo_root(PREPPED_INPUT)
assert SRC, (f"Could not find train/images under {PREPPED_INPUT}. "
             f"Contents: {os.listdir(PREPPED_INPUT) if os.path.isdir(PREPPED_INPUT) else 'PATH MISSING'}. "
             f"Attach the prepped dataset and set PREPPED_INPUT to its path.")

DST = os.environ["YOLO_DATASET_DIR"]
if not os.path.isdir(os.path.join(DST, "train", "images")):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
print("Dataset ready:", SRC, "->", DST, "| splits:", os.listdir(DST))


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")

In [ ]:
import os, json
from IPython.display import Image, display
from src.model_pipeline import rerun_optimization_and_retrain

# Which models to re-run THIS session. Each one is heavy (Phase-4 search + full retrain),
# so if a Kaggle session gets close to the time limit, run a subset here and re-run the
# notebook with the rest. Classification is faster than detection; SSDlite is the fastest detector.
MODELS = ["resnet50", "densenet121", "efficientnet_b0", "yolo", "fasterrcnn", "ssdlite"]

CLASSIFIERS = {"resnet50", "densenet121", "efficientnet_b0"}
reports = {}
for m in MODELS:
    print("\n" + "=" * 70 + f"\nRE-RUN: {m}\n" + "=" * 70, flush=True)
    if m in CLASSIFIERS:
        rep = rerun_optimization_and_retrain(
            m, population=6, iterations=3, proxy_epochs=3, final_epochs=12)
    else:
        final_epochs = 30 if m == "ssdlite" else 20
        rep = rerun_optimization_and_retrain(
            m, population=5, iterations=3, proxy_epochs=3,
            proxy_train_batches=120, yolo_fraction=0.3, final_epochs=final_epochs)
    reports[m] = rep
    print("PROMOTION    :", json.dumps(rep.get("promotion", {}), default=str))
    print("BEFORE->AFTER:", json.dumps(rep.get("phase7_comparison", {}), default=str))

# show the explainability + demo image for the last model in the list
last = reports[MODELS[-1]]
for key in ("phase6_explainability", "phase8_demo"):
    info = last.get(key, {}) or {}
    img = info.get("image") or info.get("output_image")
    if img and os.path.exists(img):
        print("\n", key, "->", img); display(Image(img))

## Download artifacts for hand-back

In [ ]:
import glob, os
print("Hand these back (Output tab):")
for p in sorted(glob.glob("/kaggle/working/artifacts/*_report_rerun.json")):
    print(" - report  :", p)
for p in sorted(glob.glob("/kaggle/working/artifacts/checkpoints/*_rerun.pt")):
    print(" - weights :", p, f"({os.path.getsize(p)/1e6:.1f} MB)")
for p in sorted(glob.glob("/kaggle/working/runs/**/weights/best.pt", recursive=True)):
    print(" - yolo    :", p, f"({os.path.getsize(p)/1e6:.1f} MB)")